*Copyright 2026 Google LLC.*

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Live API - Websockets Quickstart

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/websockets/Get_started_LiveAPI.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

**This** notebok demonstrates simple usage of the Gemini Live API.

This Notebook connects directly to the API websockets to demonstrate the the low-level details for anyone building without using an SDK.

- If you are not interested in the low-level websocket details you should read the [SDK version of this notebook](../../quickstarts/Get_started_LiveAPI.ipynb).

This notebook implements a simple turn-based chat where you send messages as text, and the model replies with audio. The API is capable of much more than that. The goal here is to **demonstrate with simple code**.

- The [Live API - Text to Text](../../quickstarts/Get_started_LiveAPI.ipynb) notebook is even simpler than this, as it doesn't deal with audio.
- The [Live API - Audio Streaming in Colab](./LiveAPI_streaming_in_colab.ipynb) demonstrates streaming audio **in Colab**.<br> It's more _fun_ than this notebook but **not optimized for readability**.
- The [Live API Audio Video to Audio python script](./Get_started_LiveAPI.py) doesn't work in colab, but provides a relatively readable implementation of audio and video streaming.

## Setup

### Install and import

In [ ]:
%pip install -q websockets

In [ ]:
import asyncio
import base64
import contextlib
import datetime
import os
import json
import wave
import itertools

from websockets.asyncio.client import connect
from IPython.display import display, Audio

### Constants

To run the following cell, your API key must be stored in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](../../quickstarts/Authentication.ipynb) for an example.

In [ ]:
from google.colab import userdata
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')

The Live API works with the `gemini-3.1-flash-live-preview` model. You need to use the `v1beta` API version.


In [ ]:
MODEL = 'models/gemini-3.1-flash-live-preview'

HOST='generativelanguage.googleapis.com'

URI = f'wss://{HOST}/ws/google.ai.generativelanguage.v1beta.GenerativeService.BidiGenerateContent?key={os.environ["GOOGLE_API_KEY"]}'

### Logging

Uncomment the `logger.setLevel` call to show the log messages

In [ ]:
import logging

logger = logging.getLogger('Bidi')
#logger.setLevel('DEBUG')

### Wave file writer

The code in this secrtion is not essential for understanding the API, feel free to skip to the next section.

The simplest way to playback the audio in Colab, is to write it outto a `.wav` file.

In [ ]:
@contextlib.contextmanager
def wave_file(filename, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        yield wf

## Main audio loop

The class below implements the interaction with the Live API.

In [ ]:
class AudioLoop:
  def __init__(self, turns, tools=None):
    self.turns = turns
    self.tools = tools or []
    self.ws = None
    self.index = 0

  async def run(self):
    logger.debug('connect')
    async with connect(URI, additional_headers={'Content-Type': 'application/json'}) as ws:
      self.ws = ws
      await self.setup()

      for text in self.turns:
        await self.send(text)
        await self.recv()

  async def setup(self):
      logger.debug("set_up")
      await self.ws.send(json.dumps({
          'setup' : {
               "model": MODEL,
               "generationConfig": {
                   "responseModalities": ["AUDIO"]
               },
               "tools": self.tools
          }
      }))
      raw_response = await self.ws.recv(decode=False)
      setup_response = json.loads(raw_response.decode('ascii'))
      logger.debug(f'Connected: {setup_response}')

  async def send(self, text):
    logger.debug('send')
    print(f"\n> {text}")

    # Send text as a realtime input message.
    msg = {
        "realtimeInput": {
            "text": text
        }
    }

    await self.ws.send(json.dumps(msg))
    logger.debug('sent')

  async def recv(self):
    # Start a new `.wav` file.
    file_name = f"audio_{self.index}.wav"
    with wave_file(file_name) as wav:
      self.index += 1

      logger.debug('receive')

      # Read chunks from the socket.
      async for raw_response in self.ws:
        response = json.loads(raw_response.decode())
        logger.debug(f'got chunk: {str(response)[:200]}')

        server_content = response.get('serverContent')
        if server_content is None:
          # The server can send other message types during a session
          # (e.g. sessionResumptionUpdate, toolCall). Skip them gracefully.
          logger.debug(f'Non-content server message: {list(response.keys())}')
          continue

        # Write audio chunks to the `.wav` file.
        model_turn = server_content.get('modelTurn')
        if model_turn is not None:
          for part in model_turn.get('parts', []):
            if 'inlineData' in part:
              pcm_data = base64.b64decode(part['inlineData']['data'])
              print('.', end='')
              logger.debug('Got pcm_data')
              wav.writeframes(pcm_data)

        # Break out of the loop once the model's turn is complete.
        if server_content.get('turnComplete'):
          logger.debug('turn_complete')
          break

    display(Audio(file_name, autoplay=True))
    await asyncio.sleep(2)


There are 4 methods worth describing here:

### `run` - The main loop

This method:

- Opens a `websocket` connecting to the Live API
- Calls the initial `setup` method
- Then iterates over the hardcoded `turns`, calling `send` and `recv` for each.

### `setup` - Initial setup

The `setup` method sends the `setup` message, and awaits the response. You shouldn't try to `send` or `recv` anything else from the model until you've gotten the model's `setup_complete` response.

The `setup` message (a `BidiGenerateContentSetup` object) is where you can set the `model`, `generation_config`, `system_instructions`, `tools` and `safety_settings`.

### `send` - Sends input text to the API

The `send` method takes a hardcoded text string, prints it, and sends it to the model as a `realtimeInput` message (`BidiGenerateContentRealtimeInput`).

### `recv` - Collects audio from the API and plays it

The `recv` method collects audio chunks in a loop and writes them to a `.wav` file. It breaks out of the loop once the model sends a `turn_complete` method, and then plays the audio.

To keep things simple in Colab it collects **all** the audio before playing it. The [python script](./Get_started_LiveAPI.py) demonstrate how to play audio as soon as you start to receive it (using `PyAudio`), and how to interrupt the model (implement input and audio playback on separate tasks).

## Run

### Example 1: simple usage with Google Search

In [ ]:
turns = ["What is the capital of France?", "Who are you?"]
await AudioLoop(turns=turns, tools=[{'google_search': {}}]).run()

### Example 2: function calling

In [ ]:
turns = ["Can you turn off the lights?"]
await AudioLoop(turns=turns, tools=[{'function_declarations': [{'name': 'turn_on_the_lights', 'description': 'Turns on the lights'}, {'name': 'turn_off_the_lights', 'description': 'Turns off the lights'}]}]).run()

In [ ]:
# Note: code_execution is not supported in the Live API.

## Next steps

<a name="next_steps"></a>

This tutorial just shows basic usage of the Live API, using the Python GenAI SDK.

- If you aren't looking for code, and just want to try multimedia streaming use [Live API in Google AI Studio](https://aistudio.google.com/live).
- If you want to see how to setup streaming interruptible audio and video using the Live API and the SDK see the Live API examples in [GitHub](https://github.com/google-gemini/gemini-live-api-examples).
- Try the [Tool use in the live API tutorial](../../quickstarts/Get_started_LiveAPI_tools.ipynb) for a walkthrough of Gemini's function calling capabilities with the Live API.
- Other Gemini examples can also be found in the [Cookbook](https://github.com/google-gemini/cookbook).
